# 4. DMA Controller (p38)
- [yt tutorial](https://www.youtube.com/watch?v=4JtZQ88x5_c&list=PLVxiWMqQvhg9FCteL7I0aohj1_YiUx1x8&index=15)
- DMA 是用來做 大量搬運   
- ex spi 如果沒有 DMA 要和 device 來回  一個一個搬運  等 Tx ready  
- DMA 則是  請幫我把 buffer → SPI_FIFO, 總共 N bytes
  - CPU：我去做別的事 DMA：我幫你慢慢搬


# 4.1 Overview 
- bus slave : 笨, CPU：給你資料, Device：好，我等你
- bus master: 我自己去 memory 拿資料

- The majority of hardware pipelines and **peripherals** within the BCM2835 are **bus masters**, enabling them to efficiently satisfy their own data requirements. This reduces the requirements of the DMA controller to **block-to-block memory transfers** and supporting some of the simpler peripherals.
  - 很多 device 已經很強（bus master）所以 DMA 不用做太多事 只需要負責：memory → memory （大塊資料搬運）
- In addition, the DMA controller provides a read only prefetch mode to allow data to be brought into the L2 cache in anticipation of its later use.
  - DMA 有一種模式：只讀資料（不寫回），提前把資料抓出來 把資料「先搬到 L2 cache」，因為：之後 CPU 會用到

- Beware that the DMA controller is direcly connected to the peripherals. Thus the DMA controller must be set-up to use the Physical (harware) addresses of the peripherals.

- The BCM2835 DMA Controller provides a total of **16 DMA channels**. Each channel operates independently from the others and is internally arbitrated onto one of the **3 system busses**. This means that the amount of bandwidth that a DMA channel may consume can be controlled by the arbiter settings.
  - 16 條 DMA 通道（channel 0 ~ 15）每一條 channel 就像：一個獨立的搬運工 可以讓它們同時做不同事
    - channel 0 -> SPI send data,   channel 2 -> memory copy ....  彼此獨立，可以同時跑
  - 但問題來了：資源不夠 雖然有 16 個 channel 但實際上只有： 3 條 system bus   會塞車  
  - 每個 DMA channel 在運作時 會被分配到 3 條 bus 的其中一條 但不是固定的，是： 由 arbiter（仲裁器）決定
  - arbiter = 交通警察 負責決定： 誰可以用 bus / 誰要等 / 誰優先
- Each DMA channel operates by loading a **Control Block (CB)** data structure from memory into internal registers. The **Control Block defines the required DMA operation**. Each Control Block can point to a further Control Block to be loaded and executed once the operation described in the current Control Block has completed. In this way a **linked list of Control Blocks** can be constructed in order to execute a **sequence of DMA operations without software intervention**.

- The DMA supports AXI read bursts to ensure efficient external SDRAM use. The DMA
control block contains a burst parameter which indicates the required **burst size** of certain
memory transfers. In general the **DMA doesn’t do write bursts**, although wide writes will be
done in **2 beat bursts** if possible.
  - AXI 是一種 bus 協定, 它支援 burst read 
  - burst （爆發傳輸）一次連續搬多筆資料，而不是一筆一筆搬   不用重新發 request 
  - 不做 burst write, since write 比較敏感, 但可以連續寫入兩次 2 beat bursts
- Memory-to-Peripheral transfers can be paced by a Data Request (DREQ) signal which is
generated by the peripheral. The DREQ signal is level sensitive and controls the DMA by
gating its AXI bus requests.
  - DMA 不是一直狂送資料，而是「看 peripheral 要不要」再送
  - 記憶體 → 裝置（例如 SPI） 的傳輸速度 可以由「裝置自己」控制
  - DREQ = Data Request 是 peripheral 發出的一個訊號：我準備好了，可以再給我資料
  - level sensitive 意思是：DREQ不是「一瞬間觸發」而是看「狀態」
    - 是level 而不是 edge trigger
  - DREQ = 1 → 一直允許 DMA 傳,  DREQ = 0 → 一直禁止 DMA 傳
  - gating its AXI bus requests  : DREQ = 1 DMA 可以發 AXI request（去 memory 拿資料）
     - DREQ = 0  → DMA 被擋住（不能拿資料）
- A peripheral can also provide a Panic signal alongside the DREQ to indicate that there is an
imminent(迫在眉睫) danger of FIFO underflow or overflow or similar critical situation. The Panic is
used to select the AXI apriority(優先事項) level which is then passed out onto the AXI bus so that it can
be used to influence **arbitration** in the rest of the system.
- The **allocation of peripherals to DMA channels is programmable**.
- The DMA can deal with **byte aligned transfers** and will minimise bus traffic by **buffering** and
**packing misaligned accesses**.
- Each DMA channel can be fully disabled via a top level power register to save power.

# 4.2 DMA Controller Registers (p39)
- The DMA Controller is comprised of several identical DMA Channels depending upon the
required configuration. Each individual DMA channel has an identical register map (although
LITE channels have less functionality and hence less registers).
- DMA Channel 0 is located at the address of 0x7E007000, **Channel 1 at 0x7E007100**, Channel 2 at 0x7E007200 and so on. Thus adjacent DMA Channels are offset by 0x100.
  - 所以 dma.h 設定 0x00007100 : channel 1    
- DMA Channel 15 however, is physically removed from the other DMA Channels and so has
a different address base of 0x7EE05000.

## 4.2.1 DMA Channel Register Address Map (p40)
- Each DMA channel has an **identical register map**, only the base address of each channel is different.
- There is a global enable register at the top of the Address map that can disable each DMA for powersaving.
- Only three registers in each channels register set are directly writeable (**CS, CONBLK_AD** and DEBUG). The other registers (TI, SOURCE_AD, DEST_AD, TXFR_LEN, STRIDE & NEXTCONBK), are automatically loaded from a Control Block data structure held in external memory.
  - 所以這個就會對應到 dma.h 的  dma_channel_regs



### 4.2.1.1 Control Block Data Structure
**Control Blocks (CB) are 8 words (256 bits)** in length and must start at a 256-bit aligned
address. The format of the CB data structure in memory, is shown below.
Each 32 bit word of the control block is automatically loaded into the corresponding 32 bit DMA control block register at the start of a DMA transfer. The descriptions of these registers also defines the corresponding bit locations in the CB data structure in memory.
- DMA 會從 memory load 這些 control block data structure

- Table 4-2 : DMA Control Block definition 
  - used in dma.h : dma_control_block

|32-bit word offset|Description               |Associated Read-only register|
|------------------|--------------------------|-----------------------------|
|0                 |Transfer Information      |TI                           |
|1                 |Source address            |SOURCE_AD                    |
|2                 |Destination Address       |DEST_AD                      |
|3                 |Transfer Length           |TXFR_LEN                     |
|4                 |2D mode Stride            |STRIDE                       |
|5                 |Next Control Block address|NEXTCONBK                    |
|6-7               |Reserved 0                |N/A                          |

- The DMA is started by writing the address of a CB structure into the **CONBLK_AD** register and then setting the ACTIVE bit. The DMA will **fetch the CB from the address set in the SCB_ADDR** field of this reg and it will load it into the read-only registers described below. It will then begin a **DMA transfer according to the information in the CB**.
- When it has **completed the current DMA transfer (length => 0)** the DMA will update the
**CONBLK_AD** register with the contents of the **NEXTCONBK register**, **fetch a new CB from** that address, and start the whole procedure once again.
- The DMA will stop (and clear the ACTIVE bit) when it has completed a DMA transfer and
the **NEXTCONBK register is set to 0x0000_0000**. It will load this value into the
**CONBLK_AD** reg and **then stop**.

- Most of the control block registers cannot be written to directly as they loaded automatically from memory. They can be read to provide status information, and to indicate the progress of the current DMA transfer. The value loaded into the NEXTCONBK register can be overwritten so that the linked list of Control Block data structures can be dynamically altered. However it is only safe to do this when the DMA is paused.

### 4.2.1.2 Register map  
- 對應 dma.h 的 dma_channel_regs/dma_control_block
- DMA address Map  
- 每個 channel 0x100 = 256 **bytes**,  control block 是 256 **bits**

|Address Offset|Register Name|Description                                 |Size|
|--------------|-------------|--------------------------------------------|----|
|0x0           |0_CS         |DMA Channel 0 Control and Status            |32  |
|0x4           |0_CONBLK_AD  |DMA Channel 0 Control Block Address         |32  |
|0x8           |0_TI         |DMA Channel 0 CB Word 0(Transfer info)      |32  |
|0xc           |0_SOURCE_AD  |DMA Channel 0 CB Word 1(Source Address)     |32  |
|0x10          |0_DEST_AD    |DMA Channel 0 CB Word 2(Destination Address)|32  |
|0x14          |0_TXFR_LEN   |DMA Channel 0 CB Word 3(Transfer Length)    |32  |
|0x18          |0_STRIDE     |DMA Channel 0 CB Word 4(2D Stride)          |32  |
|0x1c          |0_NEXTCONBK  |DMA Channel 0 CB Word 5(Next CB Address)    |32  |
|0x20          |0_DEBUG      |DMA Channel 0 Debug                         |32  |
|0x100         |1_CS         |DMA Channel 1 Control and Status            |32  |
|...           |...          |...                                         |    |
|0xe00         |14_CS        |DMA Channel 14 Control and Status           |32  |
|...           |...          |...                                         |    |
|0xe20         |14_DEBUG     |DMA Channel 14 Debug                        |32  |
|0xfe0         |INT_STATUS   |Interrupt status of each DMA channel        |32  |
|0xff0         |ENABLE       |Global enable bits for each DMA channel     |32  |

- 0xfe0, 0xff0 is also used in dma.h REGS_DMA_INT_STATUS / REGS_DMA_ENABLE

#### **CS (control and status) register**  (p47)
- 0_CS 1_CS 2_CS 3_CS 4_CS 5_CS 6_CS 7_CS 8_CS 9_CS 10_CS 11_CS 12_CS 13_CS 14_CS Register
  - Synopsis : DMA Control And Status register contains the main control and status bits for this DMA channel.
- used in dma.h

|Bit(s)|Field Name|Description               |Type|Reset|
|------|----------|--------------------------|----|-----|
|31    |Reset     |DMA Channel Reset         |W1SC|0x0  |
|30    |ABORT     |Abort DMA                 |W1SC|0x0  |
|29    |DISDEBUG  |disable debug pause signal|RW  |0x0  |
|28    |WAIT_FOR_OUTSTANDING_WRITES|         |RW  |0x0  |
|27:24 |          |Reserved 0                |    |0x0  |
|23:20 |PANIC_PRIORITY|                      |RW  |0x0  |
|19:16 |PRIORITY  |                          |RW  |0x0  |
|15:9  |          |reserve 0                 |    |     |
|8     |Error     |                          |RO  |0x0  |
|7     |          |reserve 0                 |    |     |
|6     |WAIT_FOR_OUTSTANDING_WRITES|         |RO  |0x0  |
|5     |DREQ_STOPS_DMA|                          |RO  |0x0  |
|4     |PAUSED    |                          |RO  |0x0  |
|3     |DREQ      |                          |RO  |0x0  |
|2     |INT       |                          |W1C |0x0  |
|1     |END       |                          |W1C |0x0  |
|0     |ACTIVE    |                          |RW  |0x0  |

#### DMA Control Block Address register (p50)
- 0_CONBLK_AD 1_CONBLK_AD 2_CONBLK_AD 3_CONBLK_AD 4_CONBLK_AD 5_CONBLK_AD 6_CONBLK_AD 7_CONBLK_AD 8_CONBLK_AD 9_CONBLK_AD 10_CONBLK_AD 11_CONBLK_AD 12_CONBLK_AD 13_CONBLK_AD 14_CONBLK_AD Register

|Bit(s)|Field Name|Description               |Type|Reset|
|------|----------|--------------------------|----|-----|
|31:0  |SCB_ADDR  |control block address     |RW  |0x0  |

#### DMA Transfer Information. (p50)
- 0_TI 1_TI 2_TI 3_TI 4_TI 5_TI 6_TI Register
- used in dma.h TI_....

|Bit(s)|Field Name    |Description               |Type|Reset|
|------|--------------|--------------------------|----|-----|
|31:27 |              |Reserved 0                |    |     |
|26    |NO_WIDE_BURSTS|                          |RW  |     |
|25:21 |WAITS         |                          |RW  |     |
|20:16 |PERMAP        |                          |RW  |     |
|15:12 |BURST LEN     |                          |RW  |     |
|11    |SRC_IGNORE    |                          |RW  |     |
|10    |SRC_DREQ      |                          |RW  |     |
|9     |SRC_WIDTH     |                          |RW  |     |
|8     |SRC_INC       |                          |RW  |     |
|7     |DEST_IGNORE   |                          |RW  |     |
|6     |DEST_DREQ     |                          |RW  |     |
|5     |DEST_WIDTH    |                          |RW  |     |
|4     |DEST_INC      |                          |RW  |     |
|3     |WAIT_RESP     |                          |RW  |     |
|2     |              |Reserved 0                |    |     |
|1     |TDMODE        |                          |RW  |     |
|0     |INTEN         |                          |RW  |     |



# 4.3 AXI Bursts (p63)
# 4.4 Error Handling 
# 4.5 DMA LITE Engines 
